
### 炭素構造説明変数作成



In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
g_df_desc = pd.read_csv(
    "../data_calculated/Carbon8_descriptor.csv", index_col=[0, 1])

g_df_energy0 = pd.read_csv("../data/Carbon8_Etot_spin.csv", index_col=[0])


In [ ]:
def make_celldesc(df_desc):
    """make cell descriptor from atom descriptor

    Args:
        df_desc (np.DataFrame): atom descriptor

    Returns:
        pd.DataFrame: cell descriptor
    """
    sumdata = []
    i = 0
    for id_ in df_desc.index.levels[0]:
        #    print(id_)
        df1 = df_desc.loc[id_, :]
        data = df1.values
        sumdata.append(np.sum(data, axis=0))
    df_celldesc = pd.DataFrame(
        sumdata, index=df_desc.index.levels[0], columns=df_desc.columns)
    return df_celldesc


g_df_celldesc = make_celldesc(g_df_desc)

# 規格化


def scale_df(df_):
    """normalize data

    Args:
        df_ (pd.DataFrame): data

    Returns:
        pd.DataFrame: normalized data
    """
    data = df_.values
    xscaler = MinMaxScaler()
    data = xscaler.fit_transform(data)
    df = pd.DataFrame(data, columns=df_.columns, index=df_.index)
    return df


g_df_celldesc = scale_df(g_df_celldesc)

# 説明変数の表示


def plot_df(df):
    data = df.values
    plt.figure()
    plt.plot(data)
    plt.show()


# plot_df(df_celldesc)
g_df_celldesc.plot()


In [ ]:
g_df_energy = scale_df(g_df_energy0)


polytypeのloadを行う。

In [ ]:
g_df_meta = pd.read_csv(
    "../data/carbon8_nn_atom.csv")[["key", "polytype"]].drop_duplicates().set_index("key")
g_df_meta


In [ ]:
g_df = pd.concat([g_df_celldesc, g_df_energy, g_df_meta], axis=1, sort=False)
g_df


In [ ]:
# energy順にsortしておく。
g_df.sort_values(by="Etot", inplace=True)
g_df.iloc[:10, :]  # 例えば１０件までを表示する。


最大を求めるという問題にするためにenergy*(-1)しておく。

In [ ]:
g_energy = g_df["Etot"]
g_df["minus_energy"] = -g_energy
del g_df["Etot"]

g_df.plot(y="minus_energy")


In [ ]:
g_df.reset_index(inplace=True)
g_index = np.array(g_df.columns)
g_index[0] = "key"
g_df.columns = g_index.tolist()
g_df.to_csv("../data_calculated/Carbon8_descriptor_energy.csv")


In [ ]:
g_df2 = pd.read_csv(
    "../data_calculated/Carbon8_descriptor_energy.csv", index_col=[0])
g_df2
